# Practical 5: Feature Engineering for Anomaly Detection

**Objective:** Create meaningful features for machine learning.

- Number of requests per IP
- Time between requests
- Status code frequency (e.g., many 404 -> suspicious)
- User-agent parsing (bots vs real browsers)
- URL entropy (suspicious randomness)

In [9]:
import pandas as pd
import math
from collections import Counter

df = pd.read_csv('logs/labeled_access_log.csv', parse_dates=['Date/Time'])
df = df.sort_values(['IP Address', 'Date/Time']).reset_index(drop=True)
print('Rows loaded:', len(df))
df.head()

Rows loaded: 47685


,IP Address,Date/Time,Request Type,Resource,Protocol,Status Code,Bytes Sent,Referrer,User Agent,Resource Normalized,label
0,1.0.170.238,2026-08-05 17:39:50+05:30,GET,/products,HTTP/1.1,200,6709,https://facebook.com,mozilla/5.0 (iphone; cpu iphone os 17_0 like m...,/products,benign
1,1.100.161.254,2026-08-02 09:28:10+05:30,GET,/help,HTTP/1.1,200,807,direct,mozilla/5.0 (iphone; cpu iphone os 17_0 like m...,/help,benign
2,1.104.218.20,2026-08-14 05:02:08+05:30,GET,/about,HTTP/1.1,200,4715,direct,mozilla/5.0 (macintosh; intel mac os x 10_15_7...,/about,benign
3,1.105.34.107,2026-08-10 03:12:08+05:30,GET,/products?id=1',OR '1'='1 HTTP/1.1,500,792,direct,mozilla/5.0 (x11; linux x86_64; rv:121.0) geck...,/products,sqli
4,1.106.111.1,2026-08-02 11:30:41+05:30,GET,/search?q=laptop,HTTP/1.1,200,4274,https://bing.com,mozilla/5.0 (iphone; cpu iphone os 17_0 like m...,/search,benign


## 1. Requests per IP

High-frequency IPs are more likely to be scripted/automated traffic.

In [10]:
requests_per_ip = df.groupby('IP Address')['IP Address'].transform('count')
df['requests_per_ip'] = requests_per_ip
df[['IP Address', 'requests_per_ip']].drop_duplicates().sort_values('requests_per_ip', ascending=False).head(10)

,IP Address,requests_per_ip
46483,93.89.37.44,25
30176,221.82.246.172,25
8828,139.11.49.144,25
25451,201.126.226.82,25
34663,41.203.107.144,25
37896,56.7.5.9,25
41633,72.244.182.201,24
13260,158.172.20.109,24
4629,119.9.107.210,24
14595,163.204.178.72,24


## 2. Time between requests

Very short gaps between consecutive requests from the same IP suggest a script, not a human clicking through pages.

In [11]:
df['time_since_prev_request'] = (
    df.groupby('IP Address')['Date/Time'].diff().dt.total_seconds()
)
# First request from an IP has no predecessor -> fill with a large sentinel value
# (treated as 'not rapid-fire', rather than 0 which would look suspicious)
df['time_since_prev_request'] = df['time_since_prev_request'].fillna(9999)
df[['IP Address', 'Date/Time', 'time_since_prev_request']].head(10)

,IP Address,Date/Time,time_since_prev_request
0,1.0.170.238,2026-08-05 17:39:50+05:30,9999.0
1,1.100.161.254,2026-08-02 09:28:10+05:30,9999.0
2,1.104.218.20,2026-08-14 05:02:08+05:30,9999.0
3,1.105.34.107,2026-08-10 03:12:08+05:30,9999.0
4,1.106.111.1,2026-08-02 11:30:41+05:30,9999.0
5,1.109.123.222,2026-08-11 02:10:09+05:30,9999.0
6,1.11.182.241,2026-08-02 02:59:05+05:30,9999.0
7,1.11.47.99,2026-08-14 18:43:09+05:30,9999.0
8,1.11.76.71,2026-08-09 01:53:17+05:30,9999.0
9,1.112.199.237,2026-08-07 20:16:21+05:30,9999.0


## 3. Status code frequency

For each IP, what fraction of its requests are errors (4xx/5xx)? A high error rate suggests probing (scanning for endpoints that don't exist, or exploits that get rejected).

In [12]:
df['is_error_status'] = (df['Status Code'] >= 400).astype(int)
error_rate_per_ip = df.groupby('IP Address')['is_error_status'].transform('mean')
df['error_rate_per_ip'] = error_rate_per_ip
df[['IP Address', 'Status Code', 'is_error_status', 'error_rate_per_ip']].sort_values(
    'error_rate_per_ip', ascending=False
).head(10)

,IP Address,Status Code,is_error_status,error_rate_per_ip
3,1.105.34.107,500,1,1.0
47677,99.91.166.103,404,1,1.0
47639,99.4.204.1,404,1,1.0
28,1.136.231.93,403,1,1.0
47624,99.255.122.247,403,1,1.0
47629,99.29.181.45,404,1,1.0
47630,99.30.112.203,404,1,1.0
47615,99.24.119.99,403,1,1.0
47589,99.207.130.78,403,1,1.0
8383,136.28.59.163,404,1,1.0


## 4. User-agent parsing: bot vs real browser

Real browsers self-identify with `Mozilla/5.0` plus a rendering engine token. Known scanning tools and HTTP libraries self-identify explicitly (masscan, sqlmap, curl, python-requests) — easy signal, no need for a full UA-parsing library for this basic pass.

In [13]:
BOT_MARKERS = ['masscan', 'sqlmap', 'curl/', 'python-requests', 'internetmeasurement']
BROWSER_ENGINES = ['gecko', 'applewebkit', 'chrome', 'safari', 'edge', 'firefox']

def classify_user_agent(ua: str) -> str:
    ua = str(ua).lower()
    if any(marker in ua for marker in BOT_MARKERS):
        return 'bot'
    if any(engine in ua for engine in BROWSER_ENGINES):
        return 'browser'
    return 'unknown'

df['ua_type'] = df['User Agent'].apply(classify_user_agent)
df['ua_type'].value_counts()

ua_type
browser    42338
bot         5347
Name: count, dtype: int64

## 5. URL entropy

Shannon entropy measures randomness in a string. Normal paths (`/products`, `/about`) are low-entropy, predictable text. Injected payloads and encoded traversal strings pack in unusual characters and symbol density, pushing entropy higher.

In [14]:
def shannon_entropy(s: str) -> float:
    s = str(s)
    if len(s) == 0:
        return 0.0
    counts = Counter(s)
    probs = [c / len(s) for c in counts.values()]
    return -sum(p * math.log2(p) for p in probs)

df['url_entropy'] = df['Resource'].apply(shannon_entropy)
df[['Resource', 'label', 'url_entropy']].sort_values('url_entropy', ascending=False).head(10)

,Resource,label,url_entropy
20558,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
20614,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
41181,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
27099,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
15642,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
4803,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
40920,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
15574,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
41351,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173
15537,/search?q=%27%20UNION%20SELECT%20username%2Cpa...,sqli,4.671173


## Compare average entropy by label

Sanity check: attack URLs should skew higher-entropy than benign ones.

In [15]:
df.groupby('label')['url_entropy'].mean().sort_values(ascending=False)

label
sqli              4.019649
path_traversal    3.273256
benign            2.813285
brute_force       2.584963
Name: url_entropy, dtype: float64

## Save feature-engineered dataset for Practical 6

In [16]:
feature_cols = [
    'IP Address', 'Date/Time', 'Request Type', 'Resource', 'Status Code',
    'requests_per_ip', 'time_since_prev_request', 'is_error_status',
    'error_rate_per_ip', 'ua_type', 'url_entropy', 'label'
]
df[feature_cols].to_csv('logs/features_access_log.csv', index=False)
print('Saved logs/features_access_log.csv')

Saved logs/features_access_log.csv
